In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# ================== Fixed parameters ==================
         
mu0 = 30.0
# noise_amp = 0.02
dt = 0.1
T_total = 5500          # Including transient
T_trans = 500           # Transient to discard (ms)

# NMDA connection
JN11 = 0.2609
JN22 = 0.2609
# JN12 = -0.0497
# JN21 = -0.0497

base_I0=0.3255

# JN11 = 0.26
# JN22 = 0.26
JN12 = -0.0
JN21 = -0.0 
# JN12 = -0.01
# JN21 = -0.01
# mu0 = 100.0
noise_amp = 0.007
noise_amp = 0.01

# GABA equivalent connection
# w_GABA_self = 0
base_w_GABA_self = 0.

base_w_GABA = 0



T_nmda = 100.0
T_ampa = 2.0
T_gaba= 10.0
gamma = 0.641
JAext = 0.00052

# F‑I parameters
a, b, d = 270, 108, 0.1540


# Sliding window(For trajectory smoothing)
time_wind = int(5/dt)
slide_wind = int(20/dt)
n_steps = int(T_total/dt) + time_wind

# ================== Input-Output function ==================
def H(x):
    arg = a * x - b
    
    denom = 1.0 - np.exp(-d * arg)
    denom = np.where(np.abs(denom) < 1e-6, 1e-6, denom)
    return arg / denom


def r_I(r1,r2,S1,S2):
    
    NMDA_to_I=1
   
    base=0.
    I=(r1+r2)+(S1+S2)*NMDA_to_I

    beta=2
    thr=20
    k=20
    return k/(1+np.exp(-beta*(I-thr)))+base 



def r_E(x):
    arg = a * x - b
    
    denom = 1.0 - np.exp(-d * arg)
    denom = np.where(np.abs(denom) < 1e-6, 1e-6, denom)

    r=arg / denom
    # if r<0:
    #     r=0
    return r


def r_I(x):
    
   
    base=0.
    
    beta=2
    thr=20
    k=20
    return k/(1+np.exp(-beta*(x-thr)))+base 

# tau_VE=3
tau_VE=2
def update_VE(V,x): 
   k=1
   ret = V + dt/tau_VE * (-V+k*x)
    
#    ret = x
   return ret
# tau_VI=3
tau_VI=2
def update_VI(V,x): 
   k=1
   ret = V + dt/tau_VI * (-V+k*x)
#    ret = x
   return ret


NMDA_to_I=1

stim_on=500

stim_off=9999

w_pre=1100
w_post=800
# ================== Single simulation ==================
def run_trial(add,coh=0, seed=None):
    w_GABA = base_w_GABA + add
    # w_GABA_self = base_w_GABA_self +add+0.2
    # w_GABA_self = base_w_GABA_self + -0.4
    I0 = base_I0
    np.random.seed(seed)
    S1, S2 = 0.1, 0.1
    SA1, SA2 = 0.0, 0.0
    SG1, SG2 = 0.0, 0.0

    SG=0.0

    rt=None
    thr=17
    # thr=15

    decision_made = False
    r1, r2 = 2.0, 2.0
    I_noise1 = np.random.normal(0, noise_amp)
    I_noise2 = np.random.normal(0, noise_amp)

    trace_r1 = np.zeros(n_steps)
    trace_r2 = np.zeros(n_steps)
    trace_rI = np.zeros(n_steps)

    trace_V1 = np.zeros(n_steps)
    trace_V2 = np.zeros(n_steps)
    trace_VI = np.zeros(n_steps)

    V_E1,V_E2,V_I=0,0,0
    for t in range(n_steps):
        time_now = t * dt
        # No stimulus, Spontaneous activity
        if time_now>T_trans+stim_on and time_now<T_trans+stim_off and not decision_made:
            I_stim1 = JAext * mu0 * (1 + coh/100)
            I_stim2 = JAext * mu0 * (1 - coh/100)
        else:
            I_stim1 = 0
            I_stim2 = 0

        # x1 = (JN11 * S1 + JN12 * S2 +
        #       w_GABA_self * SG1 + w_GABA * SG2 +
        #       I0 + I_stim1 + I_noise1)
        # x2 = (JN22 * S2 + JN21 * S1 +
        #       w_GABA_self * SG2 + w_GABA * SG1 +
        #       I0 + I_stim2 + I_noise2)
        # x1 = (JN11 * S1 + JN12 * S2 + w_GABA * SG +
        #       I0 + I_stim1 + I_noise1)
        # x2 = (JN22 * S2 + JN21 * S1 + w_GABA * SG +
        #       I0 + I_stim2 + I_noise2)
        x1 = (JN11 * S1 + w_GABA * SG +
              I0 + I_stim1 + I_noise1)
        x2 = (JN22 * S2 + w_GABA * SG +
              I0 + I_stim2 + I_noise2)

        V_E1 = update_VE(V_E1,x1)
        V_E2 = update_VE(V_E2,x2)


        r1 = r_E(V_E1)
        r2 = r_E(V_E2)


        if (r1>thr or r2>thr) and not decision_made:
            decision_made = True
            rt_idx=t

        S1  += dt * (-S1/T_nmda + (1 - S1)*gamma*r1/1000)
        S2  += dt * (-S2/T_nmda + (1 - S2)*gamma*r2/1000)
        
        # SA1 += dt * (-SA1/T_ampa + r1/1000)
        # SA2 += dt * (-SA2/T_ampa + r2/1000)
        
        # SG1 += dt * (-SG1/T_gaba + r1/1000)
        # SG2 += dt * (-SG2/T_gaba + r2/1000)
        
        xI=(r1+r2)*1+(S1+S2)*NMDA_to_I
        V_I=update_VI(V_I,xI)
        ri=r_I(V_I)
        SG += dt * (-SG/T_gaba + ri/1000)
        


        I_noise1 += dt*( - I_noise1)/T_ampa + np.sqrt(dt/T_ampa)*noise_amp*np.random.randn()
        I_noise2 += dt*( - I_noise2)/T_ampa + np.sqrt(dt/T_ampa)*noise_amp*np.random.randn()

        trace_r1[t] = r1
        trace_r2[t] = r2
        trace_rI[t] = ri

        trace_V1[t] = V_E1
        trace_V2[t] = V_E2
        trace_VI[t] = V_I

    if not decision_made:
       rt_idx=t
    w1=int(w_pre/dt)
    w2=int(w_post/dt)
    
    r1_steady = trace_r1[rt_idx-w1:rt_idx+w2]
    r2_steady = trace_r2[rt_idx-w1:rt_idx+w2]
    rI_steady = trace_rI[rt_idx-w1:rt_idx+w2]

    V1_steady = trace_V1[rt_idx-w1:rt_idx+w2]
    V2_steady = trace_V2[rt_idx-w1:rt_idx+w2]
    VI_steady = trace_VI[rt_idx-w1:rt_idx+w2]
    # r1_steady = trace_r1
    # r2_steady = trace_r2
    # rI_steady = trace_rI
    # return r1_steady,r2_steady,rI_steady,rt_idx*dt-T_trans
    return V1_steady,V2_steady,VI_steady,rt_idx*dt-T_trans-stim_on
# run_trial(0)

In [ ]:
def get_raw_power(p,coh,t_0,t_e,wind,step,n_rep=5):
    
    t_start=np.arange(t_0, t_e, step)
    t_end=t_start+wind

    l_power_all=[[] for _ in range(n_rep)]
    from tqdm import trange
    r=0
    while r<n_rep:
        v1,v2,_,rt = run_trial(p,coh)
        if rt<500 or rt>3500:
            print('剔除：',rt,p)
            continue
        time=np.arange(0, v1.shape[0])*dt
        for i in range(len(t_start)):
            # Power spectrum
            nperseg = int(200 / dt)
            f, Pxx = welch((v1+v2)[(time>t_start[i])&(time<t_end[i])], fs=1000/dt,nperseg=nperseg )
        
            l_power_all[r].append(Pxx)
            
            
        r+=1
         
    return np.array(l_power_all),time,f
    

def get_peak_power(raw_power,f,freq_range,plot=False):

    from specparam import SpectralModel

    # 1. Initialize the model
    fm = SpectralModel(
                       peak_width_limits=[20, 999],
                       min_peak_height=0.1,               # Minimum peak height
                    #    min_peak_height=0.,               # Minimum peak height
                    #    max_n_peaks=4,                    # Maximum number of fitted peaks
                       verbose=True,
                    #    aperiodic_mode='knee',
                    #    aperiodic_mode='knee',
                       )

    # 2. Fit the model
    
    fm.fit(f, raw_power,
           freq_range=freq_range,
           )  

    peak_power=fm.get_data('peak')
    # fm.fit(f, raw_power, freq_range=[1, 100])  
    
    # 2. Extract the fitted aperiodic(1/f)baseline
    # Note: _calc_aperiodic is an internal method, it is safer to use fm._calc_aperiodic(f)
    # aperiodic_fit = fm.get_data('aperiodic')
    
    # 3. Subtract to obtain the oscillatory residual(Remove negative values, small fluctuations may become negative after baseline subtraction)
    # peak_power = raw_power - aperiodic_fit
    # oscillatory_spectrum = np.clip(oscillatory_spectrum, 0, None)  # Set negative values to zero
    if plot:
        fm.plot()
    return peak_power
  

In [ ]:

# power_all,time,f=get_raw_power(-2,coh,t_start,t_end,wind,step,1)
# get_peak_power(np.mean(power_all[:,7],0),f,freq_range,plot=True)

In [ ]:

# t_end=1500
# t_start=500
t_start=0
t_end=1601
wind=200
step=50
freq_range=(10,150)
t_wind_start=np.arange(t_start, t_end, step)
# n_rep=100
coh=0.

# power_all,time,f=get_raw_power(-2,coh,t_start,t_end,wind,step,n_rep)
def get_all_results(p_values=[-0.2,-0.1],plot=True,n_rep=10):
# def get_all_results(p_values=[-2,-1],plot=True,n_rep=10):
    all_results=[]


    for p in p_values:
        power_all,time,f=get_raw_power(p,coh,t_start,t_end,wind,step,n_rep)
        peak_powers=[]
        for t in range(power_all.shape[1]):
            peak_powers.append(get_peak_power(np.mean(power_all[:,t],0),f,freq_range))
         
        peak_powers=np.array(peak_powers).T

        all_results.append(peak_powers)

        if plot:
            plt.figure(figsize=(6,6))


            import seaborn as sns

            extent = [t_wind_start[0]+wind/2-w_pre,t_wind_start[-1]+wind/2-w_pre, freq_range[0],freq_range[1]]

            im = plt.imshow(peak_powers, aspect='auto', origin='lower', extent=extent,
                            cmap='coolwarm',
                        )
            plt.colorbar()


            v1,v2,rI,_ = run_trial(p)
            x=np.linspace(-1000,500,len(rI))
            plt.figure()
            plt.plot(x,v1)
            plt.plot(x,v2)
            plt.twinx()
            plt.plot(x,rI,alpha=0.5)


    return all_results
            
    # pass
all_results=get_all_results()

In [ ]:
delt=all_results[1]-all_results[0]
ave=(all_results[1]+all_results[0])/2

m=np.max(np.abs(delt))
plt.figure(figsize=(6,3))
import seaborn as sns
extent = [t_wind_start[0]+wind/2-w_pre,t_wind_start[-1]+wind/2-w_pre, freq_range[0],freq_range[1]]

im = plt.imshow(delt, aspect='auto', origin='lower', extent=extent,
                cmap='coolwarm',vmin=-m, vmax=m,
                )
plt.colorbar()
# plt.gca().spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.xlabel('Time relative to response (ms)')
plt.ylabel('Frequency (Hz)')
plt.title('Weak inhibition minus Strong inhibition')

plt.ylim(20,120)
# plt.savefig('difference.svg')



plt.figure(figsize=(6,3))
import seaborn as sns

im = plt.imshow(ave, aspect='auto', origin='lower', extent=extent,
                cmap='coolwarm'
                )
plt.colorbar()
# plt.gca().spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.xlabel('Time relative to response (ms)')
plt.ylabel('Frequency (Hz)')
plt.title('Average')
plt.ylim(20,120)
# plt.savefig('average.svg')






In [ ]:

power_all,time,f=get_raw_power(-1,coh,t_start,t_end,wind,step,1)
l_delt=[]
l_ave=[]
l_gamma=[]
from tqdm import trange,tqdm
l_results=[]

n_level=40

# inhibition_strength=np.linspace(-2,-1,n_level)
# inhibition_strength=np.linspace(-0.2,-0.1,n_level)
inhibition_strength=np.linspace(-0.2,-0.1,n_level)
# all_results=get_all_results(p_values=inhibition_strength,plot=False)
# all_results=[get_all_results(p_values=[p],plot=False,n_rep=100)[0] for p in tqdm(inhibition_strength)]
# all_results=[get_all_results(p_values=[p],plot=False,n_rep=100)[0] for p in tqdm(inhibition_strength)]
from joblib import Parallel, delayed
# all_results=[get_all_results(p_values=[p],plot=False,n_rep=100)[0] for p in tqdm(inhibition_strength)]
all_results=Parallel(n_jobs=8)(delayed(get_all_results)(p_values=[p],plot=False,n_rep=100) for p in tqdm(inhibition_strength))

  

used_f=f[(f>freq_range[0])&(f<freq_range[1])]

In [ ]:
from scipy.io import savemat
savemat('40_level_data.mat', {'data': np.array(all_results)[:,0],'inhibition_strength':inhibition_strength,'f':used_f,'t_wind_start':t_wind_start-w_pre,'window_length':200,'window_step':50})

In [ ]:
np.array(all_results).shape

In [ ]:
from scipy.io import loadmat
mat_file = loadmat('40_level_data.mat')
# mat_file = loadmat('40_level_data(1).mat')
all_results=mat_file['data']
inhibition_strength=mat_file['inhibition_strength'].flatten()
used_f=mat_file['f'].flatten()
t_wind_start=mat_file['t_wind_start'].flatten()+w_pre


power=[]
f_idx=(used_f>40)&(used_f<70)
t_idx=(t_wind_start+wind/2<w_pre-300)&(t_wind_start+wind/2>w_pre-500)
for i in range(len(all_results)):
  

  power.append(np.mean(all_results[i][f_idx][:,t_idx]))
sns.scatterplot(x=power,y=inhibition_strength)
plt.xlabel('log(Power)')
plt.ylabel('Inhibition strength')

In [ ]:
from scipy.stats import pearsonr
(pearsonr(power,inhibition_strength))
# power=np.array(power)
# (pearsonr(power[inhibition_strength<-0.3],inhibition_strength[inhibition_strength<-0.3]))

In [ ]:

power=np.array(power)
m=int(len(all_results)/2)


idx1=np.arange(20)
idx2=np.arange(m, m+20)
np.random.shuffle(idx1)
np.random.shuffle(idx2)

(pearsonr(power[idx1]-power[idx2],inhibition_strength[idx1]-inhibition_strength[idx2]))

# idx1=np.arange(20)
# idx2=np.arange(m, m+20)
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
ttest_ind(power[idx1], power[idx2])
print(ttest_rel(power[idx1], power[idx2]))

In [ ]:
m=int(len(all_results)/2)
delt=np.mean(all_results[-m:],0)-np.mean(all_results[:m],0)
ave=(np.mean(all_results[-m:],0)+np.mean(all_results[:m],0))/2

m=np.max(np.abs(delt))
plt.figure(figsize=(6,3))
import seaborn as sns
extent = [t_wind_start[0]+wind/2-w_pre,t_wind_start[-1]+wind/2-w_pre, freq_range[0],freq_range[1]]

im = plt.imshow(delt, aspect='auto', origin='lower', extent=extent,
                cmap='coolwarm',vmin=-m, vmax=m,
                )
plt.colorbar()
# plt.gca().spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.xlabel('Time relative to response (ms)')
plt.ylabel('Frequency (Hz)')
plt.title('Weak inhibition minus Strong inhibition')

plt.ylim(20,120)
# plt.savefig('difference.svg')



plt.figure(figsize=(6,3))
import seaborn as sns

im = plt.imshow(ave, aspect='auto', origin='lower', extent=extent,
                cmap='coolwarm'
                )
plt.colorbar()
# plt.gca().spines[['top', 'right', 'bottom', 'left']].set_visible(False)
plt.xlabel('Time relative to response (ms)')
plt.ylabel('Frequency (Hz)')
plt.title('Average')
plt.ylim(20,120)
# plt.savefig('average.svg')




